<a href="https://colab.research.google.com/github/LevramUzo/gracce-groundwater-nigeria/blob/main/notebooks/01_data_acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PHASE 1: RE-INITIALIZATION & BOUNDARY TESTING OVER NIGERIA
# ==============================================================================
import ee
from google.colab import userdata

try:
    # 1. Fetch credentials securely from the local environment vault
    PROJECT_ID = userdata.get('GEE_PROJECT_ID')

    # 2. Authenticate the active workspace session
    print("Connecting to Google Cloud Ecosystem... Allow permission via pop-up.")
    ee.Authenticate()

    # 3. Mount the engine to your cloud profile
    ee.Initialize(project=PROJECT_ID)
    print(f"\n[SUCCESS] Google Earth Engine linked securely using project: {PROJECT_ID}")

    # 4. Load the Global International Boundaries database and isolate Nigeria
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria_boundary = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    # 5. Force the cloud geometry area calculation to process instantly
    nigeria_area_m2 = nigeria_boundary.geometry().area(maxError=1)
    nigeria_area_km2 = nigeria_area_m2.divide(1e6).getInfo()

    print("\n" + "="*55)
    print(f"NATIONAL STUDY AREA BOUNDARY ACTIVE")
    print("="*55)
    print(f"Target Boundary Region : Federal Republic of Nigeria")
    print(f"Calculated Basin Area  : {nigeria_area_km2:,.2f} km²")
    print("="*55 + "\n")

except Exception as e:
    print(f"\n[ERROR]: Setup failed at re-initialization.")
    print(f"Details: {e}")


Connecting to Google Cloud Ecosystem... Allow permission via pop-up.


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python



[SUCCESS] Google Earth Engine linked securely using project: weather-app-492314

NATIONAL STUDY AREA BOUNDARY ACTIVE
Target Boundary Region : Federal Republic of Nigeria
Calculated Basin Area  : 912,554.10 km²



In [1]:
# ==============================================================================
# STEP 2.2: HYDROCLIMATIC PREDICTOR EXTRACTION (CHIRPS & MODIS LST)
# ==============================================================================
import ee
import pandas as pd

try:
    # 1. Reference our Nigeria boundary layer from the first cell
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    # Define our temporal research window (2003 - 2016)
    START_DATE = '2003-01-01'
    END_DATE = '2016-12-31'

    print("Step 1/3: Accessing Climate Catalogs inside Earth Engine...")

    # 2. Ingest the CHIRPS High-Resolution Precipitation Dataset
    chirps_collection = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
        .filterDate(START_DATE, END_DATE) \
        .select('precipitation')

    # 3. Ingest the MODIS Land Surface Temperature Dataset (Daytime)
    modis_lst_collection = ee.ImageCollection("MODIS/061/MOD11A1") \
        .filterDate(START_DATE, END_DATE) \
        .select('LST_Day_1km')

    # 4. Define an extraction function to process monthly blocks over Nigeria
    def extract_monthly_climate(year, month):
        # Establish month-specific date bounds
        start = ee.Date.fromYMD(year, month, 1)
        end = start.advance(1, 'month')

        # Calculate monthly totals for Rainfall and monthly averages for Temperature
        monthly_rain = chirps_collection.filterDate(start, end).sum()
        monthly_temp = modis_lst_collection.filterDate(start, end).mean()

        # Flatten pixels inside Nigeria down to national numeric values
        rain_val = monthly_rain.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=nigeria.geometry(), scale=5500, maxPixels=1e9
        ).get('precipitation')

        temp_val = monthly_temp.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=nigeria.geometry(), scale=5500, maxPixels=1e9
        ).get('LST_Day_1km')

        # Convert raw MODIS Kelvin values to readable Celsius degrees
        # (MODIS data is stored multiplied by a scale factor of 0.02)
        celsius_val = ee.Number(temp_val).multiply(0.02).subtract(273.15)

        # Return a structured record containing calendar keys
        date_str = start.format('YYYY-MM')
        return ee.Feature(None, {
            'date': date_str,
            'Rainfall_mm': rain_val,
            'LST_Celsius': ee.Algorithms.If(temp_val, celsius_val, None)
        })

    # 5. Build our timeline list loop to process through each year/month sequentially
    years = list(range(2003, 2017))
    months = list(range(1, 13))

    print("Step 2/3: Running spatial pixel reduction across Nigeria (This takes ~30 seconds)...")
    results_list = []

    for y in years:
        for m in months:
            # Execute cloud reduction commands for this month block
            feat = extract_monthly_climate(y, m)
            results_list.append(feat)

    # Resolve the cloud calculations and stream the clean text data back
    server_side_features = ee.FeatureCollection(results_list).getInfo()

    # 6. Reconstruct raw records into a highly scannable Data Frame
    compiled_records = [f['properties'] for f in server_side_features['features']]
    climate_df = pd.DataFrame(compiled_records)
    climate_df = climate_df.dropna().sort_values('date').reset_index(drop=True)

    print("\n" + "="*65)
    print("SUCCESS: NIGERIA CLIMATE MATRIX COMPILED")
    print("="*65)
    print(climate_df.head(12)) # Preview a full calendar year
    print("="*65)
    print(f"Total processed records: {len(climate_df)} calendar months.")
    print("="*65 + "\n")

except Exception as e:
    print(f"\n[ERROR]: Climate feature extraction failed.")
    print(f"Details: {e}")



[ERROR]: Climate feature extraction failed.
Details: Earth Engine client library not initialized. See http://goo.gle/ee-auth.
